%md
#### 1. Create the table: **sme_p_business_plan_strategy** in Pre-Landed Layer

In [0]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import trim

In [0]:
def load_landed_internal_delta_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

In [0]:
run_date = '2026-02-05'
date_filter = f"businessdate = {run_date!r}"
catalog_landed = '`d-cdh`.landed'
cluster_columns = ["BusinessDate"]
landed_internal_table_path = f"{catalog_landed}.sme_p_business_plan_strategy"

In [0]:
# ============================== business_plan_strategy_management ============================

business_plan_strategy_management_extra_filters = None

if business_plan_strategy_management_extra_filters is not None:
    business_plan_strategy_management_extra_filters.append(date_filter)
    business_plan_strategy_management_final_filters = " AND ".join(
        business_plan_strategy_management_extra_filters
        )
else:
    business_plan_strategy_management_final_filters = date_filter

business_plan_strategy_management_selected_fields = [
            "ID",
            "StatusID",
            "BusinessPlanID",
            "BeginMonth",
            "Comments",
            "CreationDate",
            "CreationUser",
            "LastModifiedDate",
            "LastModifiedUser",
            "StrategyID",
            "DraftedByID",
            "ProposedByID",
            "ConfidenceID",
            "BusinessDate"
        ]

business_plan_strategy_management = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeBusinessPlanStrategyManagement",
            business_plan_strategy_management_final_filters,
            business_plan_strategy_management_selected_fields,
        )

# ============================== business_plan_strategy_management_status ============================

business_plan_strategy_management_status_extra_filters = None

if business_plan_strategy_management_status_extra_filters is not None:
    business_plan_strategy_management_status_extra_filters.append(date_filter)
    business_plan_strategy_management_status_final_filters = " AND ".join(
        business_plan_strategy_management_status_extra_filters
        )
else:
    business_plan_strategy_management_status_final_filters = date_filter

business_plan_strategy_management_status_selected_fields = ["ID", "Label", "Priority"]

business_plan_strategy_management_status = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeRfdtBpStrategyManagementStatus",
            business_plan_strategy_management_status_final_filters,
            business_plan_strategy_management_status_selected_fields,
        )

# ============================== business_plan_drafted_by ============================

business_plan_drafted_by_extra_filters = None

if business_plan_drafted_by_extra_filters is not None:
    business_plan_drafted_by_extra_filters.append(date_filter)
    business_plan_drafted_by_final_filters = " AND ".join(
        business_plan_drafted_by_extra_filters
        )
else:
    business_plan_drafted_by_final_filters = date_filter

business_plan_drafted_by_selected_fields = ["ID", "Label"]

business_plan_drafted_by = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeRfdtBpDraftedBy",
            business_plan_drafted_by_final_filters,
            business_plan_drafted_by_selected_fields,
        )

# ============================== business_plan_strategy ============================

business_plan_strategy_extra_filters = None

if business_plan_strategy_extra_filters is not None:
    business_plan_strategy_extra_filters.append(date_filter)
    business_plan_strategy_final_filters = " AND ".join(
        business_plan_strategy_extra_filters
        )
else:
    business_plan_strategy_final_filters = date_filter

business_plan_strategy_selected_fields = ["ID", "Label"]

business_plan_strategy = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeRfdtBpStrategy",
            business_plan_strategy_final_filters,
            business_plan_strategy_selected_fields,
        )

# ============================== business_plan_proposed_by ============================

business_plan_proposed_by_extra_filters = None

if business_plan_proposed_by_extra_filters is not None:
    business_plan_proposed_by_extra_filters.append(date_filter)
    business_plan_proposed_by_final_filters = " AND ".join(
        business_plan_proposed_by_extra_filters
        )
else:
    business_plan_proposed_by_final_filters = date_filter

business_plan_proposed_by_selected_fields = ["ID", "Label"]

business_plan_proposed_by = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeRfdtBpProposedBy",
            business_plan_proposed_by_final_filters,
            business_plan_proposed_by_selected_fields,
        )

# ============================== business_plan_confidence ============================

business_plan_confidence_extra_filters = None

if business_plan_confidence_extra_filters is not None:
    business_plan_confidence_extra_filters.append(date_filter)
    business_plan_confidence_final_filters = " AND ".join(
        business_plan_confidence_extra_filters
        )
else:
    business_plan_confidence_final_filters = date_filter

business_plan_confidence_selected_fields = ["ID", "Label"]

business_plan_confidence = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeRfdtBpConfidence",
            business_plan_confidence_final_filters,
            business_plan_confidence_selected_fields,
        )

# ============================== join ============================

sme_business_plan_strategy = (
    business_plan_strategy_management.join(
        business_plan_strategy_management_status,
        business_plan_strategy_management["StatusID"] == business_plan_strategy_management_status["ID"],
        "left"
    )
    .join(
       business_plan_drafted_by,
       business_plan_drafted_by["ID"] == business_plan_strategy_management["DraftedByID"],
       "left"
    )
    .join(
        business_plan_strategy,
        business_plan_strategy["ID"] == business_plan_strategy_management["StrategyID"],
        "left"
    )
    .join(
        business_plan_proposed_by,
        business_plan_proposed_by["ID"] == business_plan_strategy_management["ProposedByID"],
        "left"
    )
    .join(
       business_plan_confidence,
       business_plan_confidence["ID"] == business_plan_strategy_management["ConfidenceID"],
       "left"
    )
)

# ========================= final landed - internal table ============================

sme_business_plan_strategy_df = sme_business_plan_strategy.select(
    business_plan_strategy_management.ID, 
    business_plan_strategy_management.BusinessPlanID,
	business_plan_strategy_management_status.Label.alias("bps_status"),
	business_plan_strategy_management_status.Priority.alias("bps_status_priority"),
    business_plan_drafted_by.Label.alias("DraftBy"),
    business_plan_strategy.Label.alias("strategy"),
    business_plan_strategy_management.BeginMonth.cast("Date"),
    business_plan_proposed_by.Label.alias("ProposedBy"),
    business_plan_confidence.Label.alias("Confidence"),
    business_plan_strategy_management.Comments,
    business_plan_strategy_management.CreationDate.cast("Timestamp"),
    business_plan_strategy_management.CreationUser,
    business_plan_strategy_management.LastModifiedDate.cast("Timestamp"),
    business_plan_strategy_management.LastModifiedUser,
    business_plan_strategy_management.StrategyID,
    business_plan_strategy_management.BusinessDate
)
